# Plaka Tespit Modeli Eğitimi - Kaggle GPU

Bu notebook, Colab GPU limiti dolduğunda Kaggle Notebook üzerinde YOLO11n plaka tespit modelini eğitmek için hazırlandı. Kaggle'da sağ panelden **Settings > Accelerator > GPU** seçilmelidir.

## 1. GPU Kontrolü

In [ ]:
!nvidia-smi

## 2. Paket Kurulumu

Kaggle'da sağ panelden **Internet** açık olmalıdır. İnternet kapalıysa `pip install` ve Roboflow indirme adımı çalışmaz.

In [ ]:
!pip install -q ultralytics roboflow easyocr opencv-python-headless pandas matplotlib

In [ ]:
from ultralytics import YOLO
import ultralytics
ultralytics.checks()

## 3. Roboflow Veri Setini İndirme

API anahtarını hücre çalışınca gizli alana yapıştırın. Anahtarı notebook içine yazmayın.

In [ ]:
from getpass import getpass
from roboflow import Roboflow

ROBOFLOW_API_KEY = getpass('Roboflow API key: ')
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace('roboflow-universe-projects').project('license-plate-recognition-rxg4e')
version = project.version(11)
dataset = version.download('yolov11')
DATA_YAML = f'{dataset.location}/data.yaml'
print(DATA_YAML)

## 4. Veri Seti Kontrolü

In [ ]:
from pathlib import Path

dataset_root = Path(dataset.location)
for split in ['train', 'valid', 'test']:
    image_count = len(list((dataset_root / split / 'images').glob('*')))
    label_count = len(list((dataset_root / split / 'labels').glob('*.txt')))
    print(split, 'images:', image_count, 'labels:', label_count)

!cat {DATA_YAML}

## 5. Kaggle Çıktı Klasörü

Kaggle'da kalıcı çıktı almak için dosyalar `/kaggle/working` altına yazılır. Eğitimden sonra **Save Version** yapılırsa çıktı dosyaları indirilebilir hale gelir.

In [ ]:
from pathlib import Path

PROJECT_DIR = '/kaggle/working/plaka_projesi/plate_runs'
RUN_NAME = 'yolo11n_plate'
Path(PROJECT_DIR).mkdir(parents=True, exist_ok=True)
print('Eğitim çıktısı buraya kaydedilecek:', PROJECT_DIR)

## 6. YOLO11n Model Eğitimi

Colab kopmaları yaşandığı için ilk güvenli deneme 30 epoch olarak ayarlandı. `save_period=5` her 5 epoch'ta ara checkpoint üretir.

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11n.pt')
train_results = model.train(
    data=DATA_YAML,
    epochs=30,
    imgsz=640,
    batch=16,
    project=PROJECT_DIR,
    name=RUN_NAME,
    patience=10,
    save_period=5,
    plots=True
)

## 7. Doğrulama ve Test

In [ ]:
from ultralytics import YOLO

best_weights = f'{PROJECT_DIR}/{RUN_NAME}/weights/best.pt'
best_model = YOLO(best_weights)
val_metrics = best_model.val(data=DATA_YAML, split='val')
test_metrics = best_model.val(data=DATA_YAML, split='test')

print('VAL mAP50:', float(val_metrics.box.map50))
print('VAL mAP50-95:', float(val_metrics.box.map))
print('TEST mAP50:', float(test_metrics.box.map50))
print('TEST mAP50-95:', float(test_metrics.box.map))

## 8. Modeli Zip Olarak Hazırlama

Bu hücre `best.pt`, `last.pt`, sonuç grafikleri ve CSV dosyalarını tek zip dosyasına koyar. Kaggle'da sağ paneldeki Output bölümünden indirilebilir.

In [ ]:
!cd /kaggle/working && zip -r plaka_projesi_ciktilar.zip plaka_projesi
!ls -lh /kaggle/working/plaka_projesi_ciktilar.zip
print('best.pt yolu:', best_weights)